In [ ]:
#| hide
from shakeqr.data import *

# shakeqr

> Arrange massively multichannel time series so that neighbouring channels
> actually resemble each other -- then learn on the result.

*"Shake-QR"*: it uses QR-style spectral reordering, and said aloud it bridges
into "shake your..." -- a nod to the blues standard and the Black Crowes
record. It shakes up your data to help you find the real signals.

Most multichannel data has an arbitrary channel order. Stock tickers are
alphabetical; sensors are numbered by install date. Any representation
learner that treats the channel axis as spatial (convolutions, patches,
images) is then fighting meaningless high-frequency structure across
channels. `shakeqr` reorders the channels so that adjacent ones are maximally
similar, evaluates how well that ordering holds up on time periods it never
saw, and then builds representations on the resulting array.

Worked example throughout: the NYSE, ~2100 common stocks over four years,
with the S&P 500 as channel 0.

## What's in the box

**`data`** -- download, clean, reorder, benchmark.

- `get_nyse_data` / `prepare` -- `[channels, days]` open/close price tensors,
  cached, gap-filled, penny stocks dropped. Device-agnostic: data goes to the
  GPU once as it comes off disk and everything downstream inherits it.
- Pairing metrics: `build_corr` (zero-lag), `build_xcorr` (max over lags,
  without the small-overlap artifact), `corr_to_dist`.
- Four orderings: `greedy_order` (nearest-neighbour chain), `spectral_order`
  (Fiedler vector), `tsp_order` / `two_opt` (open-path TSP on the distance
  matrix), `hrp_order` (Hierarchical Risk Parity quasi-diagonalization).
- Diagnostics and plots: `roughness`, `corr_stats`, `show_corr`,
  `show_score_profile`, `show_rgb` (three orderings as one RGB image).
- `run_benchmark` -- Monte Carlo cross-validation over random time windows,
  every pairing x algorithm scored in-sample and held-out, paired win rates.

**`scattering`** -- wavelet scattering along time, in plain torch.

- Morlet filter bank and FFT wavelet transform with reflection padding.
- `scattering`: order-2 scattering with the time axis kept, `[C, T]` ->
  `[C, paths, T]`. `scattering_corr` is a third pairing metric: "similar
  multi-scale volatility structure" rather than "correlated returns".
- `stack_views`: several orderings of the same array as a leading views axis.

**`hypercomplex`** -- parameterized hypercomplex layers over the views axis.

- `PHMLinear`, `PHConv1d`: weights as `sum_i kron(A_i, S_i)` with the algebra
  `A_i` learned (re-implemented from Grassucci et al.'s HyperNets, MIT). With
  the Hamilton basis frozen in, `n=4` is exactly a quaternion layer.
- `make_ae` / `fit_ae`: a conv autoencoder along the ordered-channel axis with
  the orderings as hypercomplex views, plus a real-conv baseline.
- `algebra_stats` / `show_algebra`: read the learned `A_i` -- asymmetry as
  directional influence between views, the Gram matrix for collapse.

## Install

```sh
pip install git+https://github.com/drscotthawley/shakeqr.git
```

Development (nbdev): edit the notebooks under `nbs/`, then

```sh
pip install -e ".[dev]"
nbdev-export && nbdev-test
```

## Use

```python
from shakeqr.data import *
from shakeqr.scattering import scattering, scattering_corr, stack_views
from shakeqr.hypercomplex import make_ae, fit_ae, views_to_batch, ph_algebras, show_algebra

opening, closing, tickers = get_nyse_data()            # cached after the first call
prices, returns, tk = prepare(opening, closing, tickers)
returns_w = standardize(returns)                        # whiten once, early

C = build_corr(returns_w)                               # or build_xcorr / scattering_corr
D = corr_to_dist(C)
orders = {"greedy": greedy_order(C), "spectral": spectral_order(C),
          "tsp": two_opt(D, greedy_edge_order(D)), "hrp": hrp_order(D)}
corr_stats(C, orders)

Zl = log_prices(prices)
show_rgb(Zl, [orders["hrp"], orders["tsp"], orders["spectral"]], "NYSE")

S, paths = scattering(Zl, J=6)                           # [C, paths, T]
X = views_to_batch(stack_views(Zl, list(orders.values())))   # [T, n=4, C]
ae = make_ae(n=4, f=1, h=16, z=4)
hist = fit_ae(ae, X, epochs=40)
show_algebra(ph_algebras(ae)[0])
```

Every function lives in a notebook next to its tests and its reasoning; the
rendered notebooks are the documentation.